In [ ]:
import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from sklearn.preprocessing import LabelEncoder


from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
from sklearn.decomposition import PCA

import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering

import optuna
from sklearn.metrics import silhouette_score
from pylab import bone, pcolor, colorbar, plot, show
from minisom import MiniSom

In [ ]:
data=pd.read_excel(r"Clustering_marketing_campaign.xlsx")
data

In [ ]:
pd.set_option('display.max_columns',None)

In [ ]:
data.head()

In [ ]:
data.describe(include='all')

In [ ]:
data.isnull().sum()

In [ ]:
data.drop('ID',axis=1,inplace=True)

In [ ]:
data['Dt_Customer']=pd.to_datetime(data['Dt_Customer'])
data

In [ ]:
data['year_Dt'] = data['Dt_Customer'].dt.year
data['month_Dt'] = data['Dt_Customer'].dt.month
data['day_Dt']=data['Dt_Customer'].dt.day
data

In [ ]:
data.drop('Dt_Customer',axis=1,inplace=True)

In [ ]:
for i in data.columns:
    if not pd.api.types.is_numeric_dtype(data[i]):
        labelencoder=LabelEncoder()
        data[i] = labelencoder.fit_transform(data[i])
data

In [ ]:
data['Income']=data['Income'].fillna(data['Income'].median())

In [ ]:
for i in data.columns:
    sns.boxplot(x=i, data=data)
    plt.show()

In [ ]:
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
IQR=q3-q1

Lower = q1-1.5*IQR
Upper = q3+1.5*IQR

In [ ]:
for i in data.columns:
    if data[i].nunique() > 2:
        data[i]=np.where(data[i]>Upper[i],Upper[i],data[i])
        data[i]=np.where(data[i]<Lower[i],Lower[i],data[i])
        sns.boxplot(x=i, data=data)
        plt.show()

In [ ]:
sc = MinMaxScaler(feature_range = (0, 1))
inputs_scaled = sc.fit_transform(data)
data_scaled = pd.DataFrame(inputs_scaled, columns=data.columns)

K Means

In [ ]:
wcss = []
silhouette_scores = []

for i in range(1, 10):
    kmeans = KMeans(n_clusters=i, random_state=42)  
    kmeans.fit(data_scaled)
    
    wcss_iter = kmeans.inertia_
    wcss.append(wcss_iter)
    
    print(f"Cluster {i}: WCSS = {wcss_iter:.4f}")  # Print WCSS first
    
    if i > 1:  # Calculate silhouette score only if k >= 2
        y = kmeans.predict(data_scaled)
        silhouette_avg = silhouette_score(data_scaled, y)
        silhouette_scores.append(silhouette_avg)
        print(f"Silhouette Score = {silhouette_avg:.4f}")  
    else:
        print(" (Silhouette Score not applicable for k=1)")  


In [ ]:
plt.figure(figsize = (10,8))
plt.plot(range(1, 10), wcss, marker = 'o', linestyle = '--')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('K-means Clustering')
plt.show()

In [ ]:
# Automatically find the "elbow"
knee_locator = KneeLocator(range(1, 10), wcss, curve="convex", direction="decreasing")

#Convex Curve: The plot is shaped like a gradually flattening downward curve.
#WCSS is always decreasing as we increase the number of clusters.

# Get the elbow point
optimal_k = knee_locator.elbow
print(f"The optimal number of clusters (elbow point) is: {optimal_k}")

# Plot with the elbow point marked
plt.figure(figsize=(10, 8))
plt.plot(range(1, 10), wcss, marker="o", linestyle="--", label="WCSS")
plt.axvline(optimal_k, linestyle="--", color="r", label=f"Elbow at k={optimal_k}")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("K-means Clustering - Elbow Method")
plt.legend()
plt.show()


In [ ]:
kmeans = KMeans(3, random_state=42)
kmeans.fit(data_scaled)

In [ ]:
data['Segments'] = kmeans.predict(data_scaled)
data

In [ ]:
df_segm_analysis = data.groupby(['Segments'], as_index=False).mean()
df_segm_analysis['data count'] = data[['Segments','Income']].groupby(['Segments']).count()
df_segm_analysis['data proportion'] = df_segm_analysis['data count'] / df_segm_analysis['data count'].sum()

df_segm_analysis

In [ ]:
plt.figure(figsize=(10, 8))

sns.scatterplot(x=data['Income'], y=data['Recency'], hue=data['Segments'], palette=['blue', 'red', 'yellow'], s=100, edgecolor="black")

plt.title('Segmentation K-means')
plt.xlabel("Income")
plt.ylabel("Recency")
plt.legend(title="Segments")
plt.show()

In [ ]:

pca = PCA()

pca.fit(data_scaled)

pca.explained_variance_ratio_

In [ ]:
plt.figure(figsize = (12,9))
plt.plot(range(1, data_scaled.shape[1] + 1), pca.explained_variance_ratio_.cumsum(), marker = 'o', linestyle = '--')
plt.title('Explained Variance by Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()

In [ ]:
pca = PCA(n_components = 12)

pca.fit(data_scaled)

pca.components_

In [ ]:
df_pca_comp = pd.DataFrame(data = pca.components_,
                           columns = data_scaled.columns,
                           index = ['Component 1', 'Component 2', 'Component 3',
                                    'Component 4', 'Component 5', 'Component 6',
                                   'Component 7','Component 8', 'Component 9', 'Component 10',
                                    'Component 11', 'Component 12'])
df_pca_comp

In [ ]:
scores_pca = pca.transform(data_scaled)

pca_df = pd.DataFrame(scores_pca, columns=[f'Variable_{i+1}' for i in range(scores_pca.shape[1])])

pca_df

In [ ]:
wcss_pca = []
silhouette_scores_pca = []

for i in range(1, 8):
    kmeans_pca = KMeans(n_clusters=i, random_state=42)  
    kmeans_pca.fit(pca_df)
    
    wcss_iter = kmeans_pca.inertia_
    wcss_pca.append(wcss_iter)
    
    print(f"Cluster {i}: WCSS = {wcss_iter:.4f}")  
    
    if i > 1: 
        y = kmeans_pca.predict(pca_df)
        silhouette_avg = silhouette_score(pca_df, y)
        silhouette_scores_pca.append(silhouette_avg)
        print(f"Silhouette Score = {silhouette_avg:.4f}")  
    else:
        print(" (Silhouette Score not applicable for k=1)") 

In [ ]:
knee_locator = KneeLocator(range(1, 8), wcss_pca, curve="convex", direction="decreasing")

optimal_k = knee_locator.elbow
print(f"The optimal number of clusters (elbow point) is: {optimal_k}")

plt.figure(figsize=(10, 8))
plt.plot(range(1, 8), wcss_pca, marker="o", linestyle="--", label="WCSS")
plt.axvline(optimal_k, linestyle="--", color="r", label=f"Elbow at k={optimal_k}")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("K-means Clustering - Elbow Method")
plt.legend()
plt.show()


In [ ]:
kmeans_pca = KMeans(2, random_state=42)
kmeans_pca.fit(pca_df)

In [ ]:
data['Segments_pca'] = kmeans_pca.predict(pca_df)
data

In [ ]:
data = pd.concat([data,pca_df], axis=1)

data

In [ ]:
plt.figure(figsize=(10, 8))

sns.scatterplot(x=data['Variable_2'], y=data['Variable_1'], hue=data['Segments_pca'], palette=['blue', 'red'], s=100, edgecolor="black")

plt.title('Segmentation K-means')
plt.xlabel("Variable_2")
plt.ylabel("Variable_1")
plt.legend(title="Segments_pca")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))  
plt.title("Dendrograms")  

dend = shc.dendrogram(shc.linkage(data_scaled, method='ward'))

# Label heights
for x, y in zip(dend['icoord'], dend['dcoord']):
    plt.text((x[1] + x[2]) / 2, y[1], f"{y[1]:.2f}", ha='center', va='bottom', color='red')

plt.xlabel("Cluster Index")
plt.ylabel("Height")
plt.show()


In [ ]:
cluster = AgglomerativeClustering(n_clusters=2,linkage='ward')  
y = cluster.fit_predict(data_scaled)

y

In [ ]:
data['segment_HC'] = y

data

In [ ]:
silhouette_avg = silhouette_score(data_scaled, y)
print("Silhouette Score:", silhouette_avg)

In [ ]:
plt.figure(figsize=(10, 8))

sns.scatterplot(x=data['Income'], y=data['Recency'], hue=data['segment_HC'], palette=['blue', 'red'], s=100, edgecolor="black")

plt.title('Segmentation HC')
plt.xlabel("Income")
plt.ylabel("Recency")
plt.legend(title="Segments_HC")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))  
plt.title("Dendrograms")  

dend = shc.dendrogram(shc.linkage(pca_df, method='ward'))

# Label heights
for x, y in zip(dend['icoord'], dend['dcoord']):
    plt.text((x[1] + x[2]) / 2, y[1], f"{y[1]:.2f}", ha='center', va='bottom', color='red')

plt.xlabel("Cluster Index")
plt.ylabel("Height")
plt.show()


In [ ]:
cluster = AgglomerativeClustering(n_clusters=2, linkage='ward')  
y_pca = cluster.fit_predict(pca_df)

In [ ]:
data['segment_HC_pca'] = y_pca

data

In [ ]:
plt.figure(figsize=(10, 8))

sns.scatterplot(x=data['Variable_2'], y=data['Variable_1'], hue=data['segment_HC_pca'], palette=['red', 'yellow'], s=100, edgecolor="black")

plt.title('Segmentation HC PCA')
plt.xlabel("Variable_2")
plt.ylabel("Variable_1")
plt.legend(title="Segment_HC_pca")
plt.show()

In [ ]:
def objective(trial):

    x = trial.suggest_int("x", 10, 20)  # Grid size x
    y = trial.suggest_int("y", 10, 20)  # Grid size y
    sigma = trial.suggest_float("sigma", 0.1, 2.0, log=True)  # Spread of the neighborhood
    learning_rate = trial.suggest_float("learning_rate", 0.01, 1.0, log=True)  # Learning rate
    num_iteration = trial.suggest_int("num_iteration", 500, 5000)  # Training iterations

    # Initialize SOM
    som = MiniSom(x=x, y=y, input_len=inputs_scaled.shape[1], sigma=sigma, learning_rate=learning_rate, random_seed=42)
    som.random_weights_init(inputs_scaled)
    som.train_random(data=inputs_scaled, num_iteration=num_iteration)

    # Assign each input to the closest SOM neuron
    labels = np.array([som.winner(sample) for sample in inputs_scaled])

    # Flatten the neuron indices for silhouette_score
    flattened_labels = [i[0] * y + i[1] for i in labels]

    # Evaluate clustering using silhouette score (higher is better)
    try:
        score = silhouette_score(inputs_scaled, flattened_labels)
    except ValueError:
        score = -1  # Assign a negative score if clustering fails

    return score

# Run the optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# Display the best parameters
print("Best parameters:", study.best_params)
print("Best silhouette score:", study.best_value)

In [ ]:
best_params = study.best_params
best_params

In [ ]:
# Initialize SOM with the best parameters
som_opt = MiniSom(x=best_params['x'], y=best_params['y'], input_len=inputs_scaled.shape[1],
              sigma=best_params['sigma'], learning_rate=best_params['learning_rate'], random_seed=42)

som_opt.random_weights_init(inputs_scaled)

som_opt.train_random(data=inputs_scaled, num_iteration=best_params['num_iteration'])


In [ ]:
# Original Plot Code
bone()
pcolor(som_opt.distance_map().T)  # Display the distance map
colorbar()
show()


In [ ]:
distance_map = som_opt.distance_map()

threshold = 0.9

white_nodes = [(i, j) for i in range(distance_map.shape[0]) for j in range(distance_map.shape[1]) if distance_map[i, j] >= threshold]

print("Positions of Nodes (Distance >= 0.9):")
for node in white_nodes:
    print(f"Node {node} - Distance: {distance_map[node[0], node[1]]:.2f}")


In [ ]:
mappings = som_opt.win_map(inputs_scaled)

new = []

for node in white_nodes:
    if node in mappings:
        new_data = mappings[node]
        new.append(new_data)

new= np.concatenate(new, axis=0)

new = sc.inverse_transform(new)

new

In [ ]:
new_df = pd.DataFrame(new, columns=data_scaled.columns)

new_df

In [ ]:
winners = [som_opt.winner(sample) for sample in inputs_scaled]

df1 = data.copy()
df1["col"] = [1 if w in white_nodes else 0 for w in winners]

df1